In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.aml_bronze.raw_transactions")

df = df.withColumn(
    "transaction_id",
    F.sha2(
        F.concat_ws("||",
            F.col("timestamp"),
            F.col("from_account"),
            F.col("to_account"),
            F.col("amount_paid")
        ),
        256
    )
)

df.select("transaction_id", "timestamp", "from_account", "to_account").show(5, truncate=False)

In [0]:

cutoff_date = "2022/09/10 00:00"

initial_batch = df.filter(F.col("timestamp") < cutoff_date)

print(f"Initial batch: {initial_batch.count():,} rows")

(
    initial_batch.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.aml_bronze.raw_transactions_incremental")
)

In [0]:
new_batch = df.filter(F.col("timestamp") >= cutoff_date)

print(f"New batch (incremental): {new_batch.count():,} rows")

(
    new_batch.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.aml_bronze.raw_transactions_incremental")
)

# Επιβεβαίωση: το table τώρα πρέπει να έχει ΟΛΕΣ τις γραμμές
total = spark.table("workspace.aml_bronze.raw_transactions_incremental").count()
print(f"Total rows after incremental append: {total:,}")

In [0]:
(
    new_batch.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.aml_bronze.raw_transactions_incremental")
)

total_after_rerun = spark.table("workspace.aml_bronze.raw_transactions_incremental").count()
print(f"Total after re-running append: {total_after_rerun:,}")

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, "workspace.aml_bronze.raw_transactions_incremental")

(
    target_table.alias("target")
    .merge(
        new_batch.alias("source"),
        "target.transaction_id = source.transaction_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

total_after_merge = spark.table("workspace.aml_bronze.raw_transactions_incremental").count()
print(f"Total after MERGE: {total_after_merge:,}")

In [0]:
current_total = spark.table("workspace.aml_bronze.raw_transactions_incremental").count()
print(f"Current total (με duplicates): {current_total:,}")

distinct_total = (
    spark.table("workspace.aml_bronze.raw_transactions_incremental")
    .select("transaction_id")
    .distinct()
    .count()
)
print(f"Distinct transaction_ids: {distinct_total:,}")

print(f"Duplicates: {current_total - distinct_total:,}")

In [0]:
clean_df = initial_batch.unionByName(new_batch).dropDuplicates(["transaction_id"])

print(f"Clean row count: {clean_df.count():,}")

(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.aml_bronze.raw_transactions_incremental")
)

final_check = spark.table("workspace.aml_bronze.raw_transactions_incremental").count()
print(f"Final table count: {final_check:,}")